# DMRG vs NQS — 3D toric code, L=3 OBC (N=54)

Collaborator DMRG reference (bosonic + fermionic, single-field axes) against the dual-basis NQS
(`nersc/launch_dmrg_L3_btc.sh`, weak-field wave: $|h|\le 0.2$ in tc3d convention). The reference is
already converted to tc3d conventions (`analysis/scripts/dmrg_reference.py`): $h_x^{\rm ours}=-h/2$,
$h_y^{\rm ours}=h_z^{\rm ours}=+h/2$, so the $x$ axis is plotted against $|h|$ and labelled $-h_x$.
Runs are joined by `analysis/scripts/dmrg_compare.py`; the notebook runs with no run data (DMRG lines only).
The acceptance rule `dmrg_compare.matches()` is deliberately left for the user to define.

## 1. Config

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "analysis/scripts"))
import dmrg_compare as dc

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

REF = ROOT / "results/dmrg_L3_obc/dmrg_L3_obc_reference.json"
RUN_DIRS = {"btc": "dmrg_L3_obc/btc", "ftc": "dmrg_L3_obc/ftc"}   # label -> path under ROOT/results
MODEL = ["bosonic", "fermionic"]
FIGS = ROOT / "analysis/figs"
XMAX = None   # e.g. 0.25 to zoom the plots on the weak-field wave

# (field key, x-axis label, sign so that x = sign * h_ours >= 0)
AXES = [("hx", r"$-h_x$", -1.0), ("hy", r"$h_y$", 1.0), ("hz", r"$h_z$", 1.0)]
OBS = [("sx_mean", "sx_nqs", r"$\langle\sigma^x\rangle$"), ("sy_mean", "sy_nqs", r"$\langle\sigma^y\rangle$"),
       ("sz_mean", "sz_nqs", r"$\langle\sigma^z\rangle$"), ("A_v_mean", "A_nqs", r"$\langle A_v\rangle$"),
       ("B_p_mean", "B_nqs", r"$\langle B_p\rangle$")]
ARM_COLOR = {a: plt.cm.plasma(v) for a, v in
             {"dual": 0.15, "primal": 0.65, "A": 0.15, "B": 0.5, "C": 0.8}.items()}
ARM_MARK = {"dual": "o", "primal": "s", "A": "o", "B": "s", "C": "^"}
def arm_style(arm):
    k = arm if arm in ARM_COLOR else arm[:1]
    return ARM_COLOR.get(k, "k"), ARM_MARK.get(k, "o")

## 2. Load reference and runs

In [ ]:
ref_rows = dc.load_ref(REF)
print(f"reference: {len(ref_rows)} rows from {REF.relative_to(ROOT)}")

run_dirs = []
for label, rel in RUN_DIRS.items():
    d = ROOT / "results" / rel
    if d.is_dir():
        run_dirs.append(d)
    else:
        print(f"  [{label}] {d.relative_to(ROOT)} missing -> no NQS points (DMRG only)")
TABLE = dc.load_table(run_dirs, REF) if run_dirs else []
print(f"NQS runs loaded: {len(TABLE)}  "
      + ", ".join(f"{m}={sum(r['model'] == m for r in TABLE)}" for m in MODEL))

def axis_of(o):
    # the single non-zero field of a reference/run point; h=0 belongs to every axis
    nz = [k for k in ("hx", "hy", "hz") if abs(o[k]) > 1e-12]
    return nz[0] if len(nz) == 1 else ("0" if not nz else None)

def dmrg_curve(model, axis, key):
    pts = [(r["ours"][axis], r["ours"][key]) for r in ref_rows
           if r["model"] == model and axis_of(r["ours"]) in (axis, "0")]
    pts.sort()
    return np.array([p[0] for p in pts]), np.array([p[1] for p in pts])

def nqs_points(model, axis, key, arm=None):
    rs = [r for r in TABLE if r["model"] == model and (arm is None or r["arm"] == arm)
          and axis_of(r) in (axis, "0") and r.get(key) is not None]
    rs.sort(key=lambda r: r[axis])
    return rs

def arms(model):
    return sorted({r["arm"] for r in TABLE if r["model"] == model})

## 3. Bosonic: energy vs DMRG

In [ ]:
def fig_energy(model, title):
    fig, axs = plt.subplots(3, 3, figsize=(11, 8.5))
    for i, (ax_key, xlab, sg) in enumerate(AXES):
        a_e, a_d, a_r = axs[i]
        h, E = dmrg_curve(model, ax_key, "E")
        a_e.plot(sg * h, E, "-", color="0.55", lw=1.2, zorder=1)
        a_e.plot(sg * h, E, ".", color="0.35", ms=5, zorder=2, label="DMRG")
        a_d.axhline(0, ls="--", color="0.3", lw=1)
        for arm in arms(model):
            c, mk = arm_style(arm)
            rs = nqs_points(model, ax_key, "E_nqs", arm)
            x = np.array([sg * r[ax_key] for r in rs])
            a_e.errorbar(x, [r["E_nqs"] for r in rs], [r["E_err"] or 0 for r in rs], fmt=mk, color=c,
                         ms=7, mfc="none", capsize=2, zorder=3, label=f"NQS {arm}")
            rd = [r for r in rs if r["dE"] is not None]
            a_d.errorbar([sg * r[ax_key] for r in rd], [r["dE"] for r in rd], [r["E_err"] or 0 for r in rd],
                         fmt=mk, color=c, ms=7, mfc="none", capsize=2)
            rr = [r for r in rd if r["dE"] != 0]
            a_r.plot([sg * r[ax_key] for r in rr], [abs(r["rel"]) for r in rr], mk, color=c, ms=7, mfc="none")
        a_r.set_yscale("log"); a_r.set_ylim(1e-9, 1e-1)
        if not arms(model):
            a_d.set_ylim(-1e-3, 1e-3)
        for a in (a_e, a_d, a_r):
            a.set_xlim(-0.01 * (XMAX or 1), XMAX)
        a_e.set_ylabel(r"$E$"); a_d.set_ylabel(r"$\Delta E=E_{\rm NQS}-E_{\rm DMRG}$")
        a_r.set_ylabel(r"$|\Delta E|/|E|$")
        for a in (a_e, a_d, a_r):
            a.set_xlabel(xlab)
        if i == 0:
            a_e.legend(frameon=False, fontsize=8)
    if not arms(model):
        axs[0, 1].set_title("no NQS runs loaded -- DMRG reference only", fontsize=9, color="0.4")
    fig.suptitle(title); fig.tight_layout()
    return fig

fig_energy("bosonic", "Bosonic 3D toric code, L=3 OBC")
# plt.savefig(FIGS / "dmrg_L3_bosonic_energy.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Observables vs DMRG

In [ ]:
# Only the observables that RESPOND on each axis are plotted (the others are 0 or 1 to the DMRG
# digits in both methods and would autoscale to MCMC noise): the field-aligned <sigma> and the
# stabilizer(s) the field breaks. NQS points carry the pooled final-eval error bar; the lower
# row shows the pull (NQS - DMRG)/sigma per observable.
AXIS_OBS = {"hx": [("sx", r"$\langle\sigma^x\rangle$"), ("B", r"$\langle B_p\rangle$")],
            "hy": [("sy", r"$\langle\sigma^y\rangle$"), ("A", r"$\langle A_v\rangle$"), ("B", r"$\langle B_p\rangle$")],
            "hz": [("sz", r"$\langle\sigma^z\rangle$"), ("A", r"$\langle A_v\rangle$")]}
OBS_KEY = {"sx": "sx_mean", "sy": "sy_mean", "sz": "sz_mean", "A": "A_v_mean", "B": "B_p_mean"}
OBS_MARK = {"sx": "o", "sy": "o", "sz": "o", "A": "s", "B": "D"}

def fig_obs(model, title, xmax=XMAX):
    nrow = max(len(v) for v in AXIS_OBS.values())
    fig, axs = plt.subplots(nrow + 1, 3, figsize=(11, 3.0 * (nrow + 1)),
                            gridspec_kw={"height_ratios": [1] * nrow + [0.8]})
    for j, (ax_key, xlab, sg) in enumerate(AXES):
        a_pull = axs[nrow, j]
        a_pull.axhline(0, ls="--", color="0.3", lw=1)
        for s in (-3, 3):
            a_pull.axhline(s, ls=":", color="0.6", lw=0.8)
        for i in range(nrow):
            axs[i, j].set_visible(i < len(AXIS_OBS[ax_key]))
        for i, (short, ylab) in enumerate(AXIS_OBS[ax_key]):
            a = axs[i, j]
            h, v = dmrg_curve(model, ax_key, OBS_KEY[short])
            keep = h <= (xmax if xmax else np.inf) * 1.05
            a.plot(sg * h[keep], v[keep], "-", color="0.55", lw=1.2, zorder=1)
            a.plot(sg * h[keep], v[keep], ".", color="0.35", ms=6, zorder=2, label="DMRG")
            for arm in arms(model):
                c, mk = arm_style(arm)
                rs = nqs_points(model, ax_key, short + "_nqs", arm)
                x = np.array([sg * r[ax_key] for r in rs])
                y = np.array([r[short + "_nqs"] for r in rs])
                e = np.array([r.get(short + "_err") or 0 for r in rs])
                a.errorbar(x, y, e, fmt=mk, color=c, ms=7, mfc="none", capsize=2, zorder=3, label=f"NQS {arm}")
                ok = [k for k, r in enumerate(rs) if r.get(short + "_ref") is not None and e[k] > 0]
                a_pull.plot(x[ok], [(y[k] - rs[k][short + "_ref"]) / e[k] for k in ok], OBS_MARK[short],
                            color=c, ms=7, mfc="none")
            a.set_ylabel(ylab)
            a.set_xlim(-0.01 * (xmax or 1), xmax)
            if i == 0 and j == 0:
                a.legend(frameon=False, fontsize=8)
        a_pull.set_xlim(-0.01 * (xmax or 1), xmax)
        a_pull.set_xlabel(xlab); a_pull.set_ylabel(r"(NQS$-$DMRG)/$\sigma$")
        a_pull.set_yscale("symlog", linthresh=5); a_pull.set_ylim(-1e3, 1e3)
        from matplotlib.lines import Line2D   # marker = observable; colour = arm (legend above)
        a_pull.legend([Line2D([], [], marker=OBS_MARK[s], ls="", color="0.3", mfc="none") for s, _ in AXIS_OBS[ax_key]],
                      [lab for _, lab in AXIS_OBS[ax_key]], frameon=False, fontsize=8, loc="upper left",
                      bbox_to_anchor=(1.0, 1.0) if j == 2 else None, ncol=1)
    fig.suptitle(title); fig.tight_layout()
    return fig

fig_obs("bosonic", "Bosonic observables that respond to the field, L=3 OBC", xmax=0.25)
# plt.savefig(FIGS / "dmrg_L3_bosonic_observables.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Fermionic: arms A / B / C

In [ ]:
fig_energy("fermionic", "Fermionic 3D toric code, L=3 OBC  (arms: A sign-frame, B analytic head, C blind)")
# plt.savefig(FIGS / "dmrg_L3_fermionic_energy.png", dpi=300, bbox_inches="tight")
plt.show()

fig_obs("fermionic", "Fermionic observables, L=3 OBC")
# plt.savefig(FIGS / "dmrg_L3_fermionic_observables.png", dpi=300, bbox_inches="tight")
plt.show()

try:
    import pandas as pd
    ferm = [r for r in TABLE if r["model"] == "fermionic" and r["rel"] is not None]
    if ferm:
        df = pd.DataFrame(ferm)
        df["point"] = [f"hx={r.hx:+.2f} hy={r.hy:.2f} hz={r.hz:.2f}" for r in df.itertuples()]
        print("relative error |dE|/|E| per arm per point")
        print(df.assign(rel=df.rel.abs()).pivot_table(index="point", columns="arm", values="rel").to_string())
    else:
        print("no fermionic runs with a reference match yet")
except ImportError:
    print("pandas not installed; see the verdict table below")

## 6. Verdict table

In [ ]:
if TABLE:
    print(dc.format_table(TABLE))
    n_y = sum(r["match"] is True for r in TABLE); n_n = sum(r["match"] is False for r in TABLE)
    print(f"\nmatch: {n_y} yes / {n_n} no / {sum(r['match'] is None for r in TABLE)} undecided "
          "(define dmrg_compare.matches() to enable)")
else:
    print("no NQS runs loaded yet. Reference points of the weak-field wave (ours, bosonic):")
    for r in ref_rows:
        o = r["ours"]
        if r["model"] == "bosonic" and max(abs(o[k]) for k in ("hx", "hy", "hz")) <= 0.2 + 1e-9 \
                and axis_of(o) not in ("0", None):
            print(f"  hx={o['hx']:+.2f} hy={o['hy']:.2f} hz={o['hz']:.2f}   E_DMRG={o['E']:.6f}")